# General Results
This notebook contains results that are not specific to either model but rather concern both models. Each section operates per dataset (Eedi / SciQ).

Pure local-CSV analysis (annotated traces, no API calls). Produces: Table 3 (DS-reasoning subset) and its full 4-way breakdown Table 18 (Appendix B.2); the presence/coverage table Table 32 (Appendix D.1); and the cross-model agreement numbers behind Section 4.3's "reported trends are consistent across GLM" claim (Appendix B.4).

In [ ]:
import glob
import ast
from collections import defaultdict, Counter

import pandas as pd
import numpy as np

from src.evaluation import get_mean_and_ci

In [ ]:
EEDI_LABELS = ["INST","INTER","LINK","ERR_DESC","ERR_SIM","RECON","PLAUS","DISCR","CURATE","CORR"]
SCIQ_LABELS = ["INST","INTER","LINK","ERR_DESC","ERR_SIM","RECON","PLAUS","DISCR","CURATE","CORR"]

EEDI_TAG_LABELS = [f"<{l}>" for l in EEDI_LABELS]
SCIQ_TAG_LABELS = [f"<{l}>" for l in SCIQ_LABELS]

EEDI_TAG_NAMES = [l.lower() for l in EEDI_LABELS]
SCIQ_TAG_NAMES = [l.lower() for l in SCIQ_LABELS]

TAG_DISPLAY_NAMES = {
    "inter": "Task Interpretation",
    "corr": "Correct Answer Ref.",
    "link": "Distractor Linking",
    "err_desc": "Error Description",
    "inst": "Outcome Instantiation",
    "err_sim": "Error Simulation",
    "plaus": "Plausibility Check",
    "discr": "Discrimination Check",
    "curate": "Final Set Curation",
    "recon": "Reconsideration"
}

In [ ]:
# Parameterized analysis helpers — both EEDI and SciQ sections call these.

def load_and_parse_traces(pattern, tag_names):
    """Return positions_by_tag for matching CSVs."""
    positions_by_tag = {tag: [] for tag in tag_names}
    for path in glob.glob(pattern):
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            reasoning = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            reasoning_len = len(reasoning) if reasoning else 1
            for pos, tag in seq:
                if tag in positions_by_tag:
                    positions_by_tag[tag].append(pos / reasoning_len)
    return positions_by_tag


def compute_trace_stats(pattern, tag_names):
    paths = glob.glob(pattern)
    total_traces = 0
    total_len = 0
    presence = {tag: 0 for tag in tag_names}
    counts_per_trace = {tag: [] for tag in tag_names}

    for path in paths:
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            trace = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            total_traces += 1
            total_len += len(trace) if isinstance(trace, str) else 0

            labels_list = [label for _, label in seq if label in tag_names]
            for tag in set(labels_list):
                presence[tag] += 1

            counts = Counter(labels_list)
            for tag in tag_names:
                counts_per_trace[tag].append(counts.get(tag, 0))

    avg_len = (total_len / total_traces) if total_traces > 0 else 0
    presence_pct = {tag: (presence[tag] / total_traces * 100) if total_traces > 0 else 0 for tag in tag_names}
    return total_traces, avg_len, presence, presence_pct, counts_per_trace


def print_summary_stats(label, reasoning_pattern, cot_pattern, tag_names):
    print("\n" + "="*80)
    print(f"SUMMARY STATISTICS — {label}")
    print("="*80)

    r_total, r_avg_len, _, r_presence_pct, r_counts_per_trace = compute_trace_stats(reasoning_pattern, tag_names)
    ct_total, ct_avg_len, _, ct_presence_pct, ct_counts_per_trace = compute_trace_stats(cot_pattern, tag_names)

    print("\nAverage trace lengths:")
    print(f"  Reasoning traces (n={r_total}): avg length = {r_avg_len:.1f} chars")
    print(f"  CoT traces (n={ct_total}): avg length = {ct_avg_len:.1f} chars")

    print("\nAverage occurrences per trace (mean ± 95% CI):")
    for label_name, counts in [("Reasoning traces", r_counts_per_trace), ("CoT traces", ct_counts_per_trace)]:
        print(f"  {label_name}:")
        for tag in tag_names:
            arr = np.array(counts[tag])
            if len(arr) > 1:
                mean, ci = get_mean_and_ci(arr)
            else:
                mean, ci = (np.mean(arr) if len(arr) > 0 else 0), 0
            print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {mean:.2f} ± {ci:.2f}")

    print("\nPercentage of traces containing at least one occurrence (per label):")
    for label_name, pct_dict in [("Reasoning traces", r_presence_pct), ("CoT traces", ct_presence_pct)]:
        print(f"  {label_name}:")
        for tag in tag_names:
            print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {pct_dict.get(tag, 0):.1f}%")

    print("\n" + "="*80)


def get_row_wise_transitions(pattern, tag_names):
    row_transitions = defaultdict(Counter)
    for path in glob.glob(pattern):
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            tags = [tag for _, tag in seq if tag in tag_names]
            for i in range(len(tags) - 1):
                src, tgt = tags[i], tags[i+1]
                row_transitions[src][tgt] += 1
    return row_transitions


def top_k_rowwise_dominant_agreement(trans1, trans2, k=2, threshold=0.15):
    source_tags = set(list(trans1.keys()) + list(trans2.keys()))
    overlaps = []
    for src in source_tags:
        c1 = trans1.get(src, Counter())
        c2 = trans2.get(src, Counter())
        t1 = sum(c1.values()) + 1e-10
        t2 = sum(c2.values()) + 1e-10
        top1 = [t for t, c in c1.items() if c / t1 >= threshold][:k]
        top2 = [t for t, c in c2.items() if c / t2 >= threshold][:k]
        if not top1 and not top2:
            continue
        if not top1 or not top2:
            overlaps.append(0.0)
        else:
            overlaps.append(len(set(top1) & set(top2)) / min(len(top1), len(top2)))
    return np.mean(overlaps) if overlaps else np.nan


def print_similarity(label, data_folder, tag_names, k=3, min_outgoing_mass=0.15):
    glm_r = get_row_wise_transitions(f"{data_folder}/joint_results/annotated/*openrouter*reasoner*_annot_parsed.csv", tag_names)
    glm_c = get_row_wise_transitions(f"{data_folder}/joint_results/annotated/*openrouter*cot*chat*_annot_parsed.csv", tag_names)
    ds_r = get_row_wise_transitions(f"{data_folder}/joint_results/annotated/*deepseek*reasoner*_annot_parsed.csv", tag_names)
    ds_c = get_row_wise_transitions(f"{data_folder}/joint_results/annotated/*deepseek*cot*chat*_annot_parsed.csv", tag_names)

    print("\n" + "="*80)
    print(f"ROW-WISE TOP-{k} DOMINANT AGREEMENT (>={min_outgoing_mass}) — {label}")
    print("="*80)
    print("\n=== REASONING TRACES ===")
    print(f"Row-wise top-{k} dominant agreement: {top_k_rowwise_dominant_agreement(glm_r, ds_r, k=k, threshold=min_outgoing_mass):.2f}")
    print("\n=== CoT TRACES ===")
    print(f"Row-wise top-{k} dominant agreement: {top_k_rowwise_dominant_agreement(glm_c, ds_c, k=k, threshold=min_outgoing_mass):.2f}")


## Eedi

### Load and Parse Annotated Data

In [ ]:
eedi_reasoning_pattern = "eedi_data/joint_results/annotated/*-reasoner_*_annot_parsed.csv"
eedi_cot_pattern = "eedi_data/joint_results/annotated/*-cot-*-chat_*_annot_parsed.csv"

print("Loading EEDI reasoning traces...")
eedi_reasoning_positions = load_and_parse_traces(eedi_reasoning_pattern, EEDI_TAG_NAMES)
print(f"  {sum(len(v) for v in eedi_reasoning_positions.values())} total component occurrences")

print("Loading EEDI CoT traces...")
eedi_cot_positions = load_and_parse_traces(eedi_cot_pattern, EEDI_TAG_NAMES)
print(f"  {sum(len(v) for v in eedi_cot_positions.values())} total component occurrences")

### Summary Statistics

In [ ]:
# Per-model summary stats for EEDI
for model_label, reasoning_stem, cot_stem in [
    ('DeepSeek', 'deepseek-naive-deepseek-reasoner',        'deepseek-naive-cot-deepseek-chat'),
    ('GLM',      'openrouter-naive-z-ai_glm-4.7-reasoner', 'openrouter-naive-cot-z-ai_glm-4.7-chat'),
]:
    rp = f"eedi_data/joint_results/annotated/{reasoning_stem}_*_annot_parsed.csv"
    cp = f"eedi_data/joint_results/annotated/{cot_stem}_*_annot_parsed.csv"
    print_summary_stats(f"EEDI / {model_label}", rp, cp, EEDI_TAG_NAMES)


### Similarity Between Models

In [ ]:
print_similarity("EEDI", "eedi_data", EEDI_TAG_NAMES, k=3, min_outgoing_mass=0.15)

### Per-strategy ratio (Reasoning / CoT)

In [ ]:
# Per-strategy Reasoning/CoT ratio (EEDI), derived from compute_trace_stats so it stays
# in sync with the taxonomy.

def _mean_counts(pattern, tag_names):
    _, _, _, _, counts_per_trace = compute_trace_stats(pattern, tag_names)
    return np.array([np.mean(counts_per_trace[tag]) if counts_per_trace[tag] else 0.0
                     for tag in tag_names])


DS_REASONING_STEM = "deepseek-naive-deepseek-reasoner"
DS_COT_STEM       = "deepseek-naive-cot-deepseek-chat"
GLM_REASONING_STEM = "openrouter-naive-z-ai_glm-4.7-reasoner"
GLM_COT_STEM       = "openrouter-naive-cot-z-ai_glm-4.7-chat"

ds_cot_means        = _mean_counts(f"eedi_data/joint_results/annotated/{DS_COT_STEM}_*_annot_parsed.csv",        EEDI_TAG_NAMES)
ds_reasoning_means  = _mean_counts(f"eedi_data/joint_results/annotated/{DS_REASONING_STEM}_*_annot_parsed.csv",  EEDI_TAG_NAMES)
glm_cot_means       = _mean_counts(f"eedi_data/joint_results/annotated/{GLM_COT_STEM}_*_annot_parsed.csv",       EEDI_TAG_NAMES)
glm_reasoning_means = _mean_counts(f"eedi_data/joint_results/annotated/{GLM_REASONING_STEM}_*_annot_parsed.csv", EEDI_TAG_NAMES)

strategies = [TAG_DISPLAY_NAMES.get(tag, tag) for tag in EEDI_TAG_NAMES]

eps = 1e-9
ds_ratio  = ds_reasoning_means  / np.maximum(ds_cot_means,  eps)
glm_ratio = glm_reasoning_means / np.maximum(glm_cot_means, eps)

print("Per-strategy Reasoning/CoT ratio (DS vs GLM):")
for s, d, g in zip(strategies, ds_ratio, glm_ratio):
    denom = (d + g)
    rel_diff = (2 * abs(d - g) / denom) if denom > 0 else float("nan")
    print(f"  {s}: {d:.2f} vs {g:.2f}  (rel. diff: {rel_diff:.3f})")


## SciQ

### Load and Parse Annotated Data

In [ ]:
sciq_reasoning_pattern = "sciq_data/joint_results/annotated/*-reasoner_*_annot_parsed.csv"
sciq_cot_pattern = "sciq_data/joint_results/annotated/*-cot-*-chat_*_annot_parsed.csv"

print("Loading SciQ reasoning traces...")
sciq_reasoning_positions = load_and_parse_traces(sciq_reasoning_pattern, SCIQ_TAG_NAMES)
print(f"  {sum(len(v) for v in sciq_reasoning_positions.values())} total component occurrences")

print("Loading SciQ CoT traces...")
sciq_cot_positions = load_and_parse_traces(sciq_cot_pattern, SCIQ_TAG_NAMES)
print(f"  {sum(len(v) for v in sciq_cot_positions.values())} total component occurrences")

### Summary Statistics

In [ ]:
# Per-model summary stats for SciQ
for model_label, reasoning_stem, cot_stem in [
    ('DeepSeek', 'deepseek-naive-deepseek-reasoner',        'deepseek-naive-cot-deepseek-chat'),
    ('GLM',      'openrouter-naive-z-ai_glm-4.7-reasoner', 'openrouter-naive-cot-z-ai_glm-4.7-chat'),
]:
    rp = f"sciq_data/joint_results/annotated/{reasoning_stem}_*_annot_parsed.csv"
    cp = f"sciq_data/joint_results/annotated/{cot_stem}_*_annot_parsed.csv"
    print_summary_stats(f"SciQ / {model_label}", rp, cp, SCIQ_TAG_NAMES)


### Similarity Between Models

In [ ]:
print_similarity("SciQ", "sciq_data", SCIQ_TAG_NAMES, k=3, min_outgoing_mass=0.15)

# Latex

Next cell -> Table 18; Table 3 in the main body is the DeepSeek-reasoning subset of this table, copied out by hand.

In [ ]:
# Build the LaTeX taxonomy-occurrence table (mean ± 95% CI), ready to copy.

ROW_ORDER = [
    ("inter",    "Task Interpretation"),
    ("corr",     "Correct Answer Ref."),
    ("link",     "Conceptual Link"),
    ("err_desc", "Error Description"),
    ("err_sim",  "Error Simulation"),
    ("inst",     "Outcome Instantiation"),
    ("plaus",    "Plausibility Check"),
    ("discr",    "Discriminability Check"),
    ("curate",   "Final Set Curation"),
    ("recon",    "Reconsideration"),
]

MODELS = [
    ("\\ds",  "deepseek-naive-deepseek-reasoner",       "deepseek-naive-cot-deepseek-chat"),
    ("\\glm", "openrouter-naive-z-ai_glm-4.7-reasoner", "openrouter-naive-cot-z-ai_glm-4.7-chat"),
]


def _stats_for(pattern, tag_names):
    _, _, _, _, counts_per_trace = compute_trace_stats(pattern, tag_names)
    out = {}
    for tag in tag_names:
        arr = np.array(counts_per_trace[tag])
        if len(arr) > 1:
            mean, ci = get_mean_and_ci(arr)
        else:
            mean, ci = (float(np.mean(arr)) if len(arr) > 0 else 0.0), 0.0
        out[tag] = (float(mean), float(ci))
    return out


def _split2(x):
    a, b = f"{x:.2f}".split(".")
    return a, b


def _fmt_pair(mean, ci):
    m_i, m_d = _split2(mean)
    c_i, c_d = _split2(ci)
    return f"{m_i} & {m_d} & {c_i} & {c_d}"


stats = {}
for model_name, r_stem, c_stem in MODELS:
    stats[model_name] = {
        ("cot", "Eedi"):       _stats_for(f"eedi_data/joint_results/annotated/{c_stem}_*_annot_parsed.csv", EEDI_TAG_NAMES),
        ("cot", "SciQ"):       _stats_for(f"sciq_data/joint_results/annotated/{c_stem}_*_annot_parsed.csv", SCIQ_TAG_NAMES),
        ("reasoning", "Eedi"): _stats_for(f"eedi_data/joint_results/annotated/{r_stem}_*_annot_parsed.csv", EEDI_TAG_NAMES),
        ("reasoning", "SciQ"): _stats_for(f"sciq_data/joint_results/annotated/{r_stem}_*_annot_parsed.csv", SCIQ_TAG_NAMES),
    }


header = r"""\begin{table*}[!htbp]
\centering
\small
\begin{tabular}{ll|
                r@{.}l@{\,$\pm$\,}r@{.}l
                r@{.}l@{\,$\pm$\,}r@{.}l
                r@{.}l@{\,$\pm$\,}r@{.}l
                r@{.}l@{\,$\pm$\,}r@{.}l}
\hline
& &
\multicolumn{8}{c}{\textbf{CoT}}
& \multicolumn{8}{c}{\textbf{Reasoning}} \\
\cmidrule(lr){3-10}
\cmidrule(lr){11-18}

\textbf{Model} & \textbf{Strategy} &
\multicolumn{4}{c}{\textbf{Eedi}}
& \multicolumn{4}{c}{\textbf{SciQ}}
& \multicolumn{4}{c}{\textbf{Eedi}}
& \multicolumn{4}{c}{\textbf{SciQ}} \\
\hline
"""

footer = r"""\hline
\end{tabular}
\caption{Average occurrences of taxonomy strategies (\cref{tab:taxonomy-definition}) in CoT and reasoning traces for each model on both datasets (mean $\pm$ 95\% CI based on t-distribution).}
\label{tab:component_presence_full}
\end{table*}"""

body_blocks = []
for model_name, _, _ in MODELS:
    rows = []
    for tag, display in ROW_ORDER:
        cells = []
        for trace_type in ("cot", "reasoning"):
            for dataset in ("Eedi", "SciQ"):
                m, c = stats[model_name][(trace_type, dataset)][tag]
                cells.append(_fmt_pair(m, c))
        rows.append(f"& {display}\n& " + "\n& ".join(cells) + r" \\")
    block = f"\\multirow{{10}}{{*}}{{\\textbf{{{model_name}}}}}\n\n" + "\n\n".join(rows)
    body_blocks.append(block)

body = "\n\n\\hline\n\n".join(body_blocks)

print(header + body + "\n\n" + footer)


In [ ]:
# Avg number of strategy occurrences per trace, broken down by
# model x {cot, reasoning} x {eedi, sciq}, plus aggregates that average over
# regimes / datasets / models / overall.
#
# Weighting: each (strategy, model, regime, dataset) combination contributes
# one number = mean occurrences of that strategy across traces in that cell.
# A cell's reported value is the unweighted mean of its per-strategy means.
# Aggregates are the unweighted mean of the per-strategy means across all
# included (strategy, model, regime, dataset) combinations.
# CIs are computed across the contributing per-strategy means (t-based).

def _strategy_means(pattern, tag_names):
    _, _, _, _, counts_per_trace = compute_trace_stats(pattern, tag_names)
    return {tag: (float(np.mean(counts_per_trace[tag])) if counts_per_trace[tag] else 0.0)
            for tag in tag_names}


# strategy_means[(model, regime, dataset)] -> {tag: mean occurrences per trace}
strategy_means = {}
for model_name, r_stem, c_stem in MODELS:
    pretty_model = model_name.replace('\\', '')
    for regime, stem in [("cot", c_stem), ("reasoning", r_stem)]:
        for dataset, folder, tag_names in [
            ("Eedi", "eedi_data", EEDI_TAG_NAMES),
            ("SciQ", "sciq_data", SCIQ_TAG_NAMES),
        ]:
            pattern = f"{folder}/joint_results/annotated/{stem}_*_annot_parsed.csv"
            strategy_means[(pretty_model, regime, dataset)] = _strategy_means(pattern, tag_names)


def _summary(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) > 1:
        mean, ci = get_mean_and_ci(arr)
    elif len(arr) == 1:
        mean, ci = float(arr[0]), 0.0
    else:
        mean, ci = 0.0, 0.0
    return len(arr), mean, ci


def _pool(filter_fn):
    out = []
    for (m, r, d), tag_means in strategy_means.items():
        if filter_fn(m, r, d):
            out.extend(tag_means.values())
    return out


models   = sorted({k[0] for k in strategy_means})
regimes  = ["cot", "reasoning"]
datasets = ["Eedi", "SciQ"]

print(f"{'Model':<10} {'Regime':<12} {'Dataset':<8}  k    mean +/- 95% CI")
print("  (k = number of per-strategy means contributing to the average)")
print("-" * 60)

# --- Per-cell (model x regime x dataset): mean over its strategies ---
for m in models:
    for r in regimes:
        for d in datasets:
            k, mean, ci = _summary(list(strategy_means[(m, r, d)].values()))
            print(f"{m:<10} {r:<12} {d:<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

print("-" * 60)
print("Aggregates (unweighted mean over per-strategy means):")
print("-" * 60)

# --- Aggregate over datasets (per model x regime) ---
for m in models:
    for r in regimes:
        k, mean, ci = _summary(_pool(lambda mm, rr, dd, m=m, r=r: mm == m and rr == r))
        print(f"{m:<10} {r:<12} {'ALL':<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

# --- Aggregate over regimes (per model x dataset) ---
for m in models:
    for d in datasets:
        k, mean, ci = _summary(_pool(lambda mm, rr, dd, m=m, d=d: mm == m and dd == d))
        print(f"{m:<10} {'ALL':<12} {d:<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

# --- Aggregate over models (per regime x dataset) ---
for r in regimes:
    for d in datasets:
        k, mean, ci = _summary(_pool(lambda mm, rr, dd, r=r, d=d: rr == r and dd == d))
        print(f"{'ALL':<10} {r:<12} {d:<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

# --- Aggregate over models + datasets (per regime) ---
for r in regimes:
    k, mean, ci = _summary(_pool(lambda mm, rr, dd, r=r: rr == r))
    print(f"{'ALL':<10} {r:<12} {'ALL':<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

# --- Aggregate over models + regimes (per dataset) ---
for d in datasets:
    k, mean, ci = _summary(_pool(lambda mm, rr, dd, d=d: dd == d))
    print(f"{'ALL':<10} {'ALL':<12} {d:<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

# --- Aggregate over regimes + datasets (per model) ---
for m in models:
    k, mean, ci = _summary(_pool(lambda mm, rr, dd, m=m: mm == m))
    print(f"{m:<10} {'ALL':<12} {'ALL':<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")

# --- Overall ---
k, mean, ci = _summary(_pool(lambda mm, rr, dd: True))
print(f"{'ALL':<10} {'ALL':<12} {'ALL':<8}  {k:<3}  {mean:6.2f} +/- {ci:5.2f}")


In [ ]:
# Per-strategy reasoning/CoT ratio per model, averaged across datasets,
# to check claim (ii): the ratios are of similar magnitude for both models,
# except for RECON, ERR_SIM, PLAUS where DS's ratio is significantly larger.
#
# Procedure (each dataset gives one independent ratio per strategy x model):
#   ratio_{m, s, d} = mean_occ_per_trace_reasoning(m, s, d) /
#                     mean_occ_per_trace_cot(m, s, d)
#   reported value per (model, strategy) = mean over datasets of the two ratios.
#   significance: Welch's t-test on the two-element ratio samples
#                 (DS ratios vs GLM ratios across datasets).

from scipy import stats as _sps

EPS = 1e-9
STRATEGY_DISPLAY = dict(ROW_ORDER)  # tag -> display name (defined earlier)

# strategy_means is already built in the previous cell; rebuild defensively in
# case this cell is re-run independently.
if "strategy_means" not in globals():
    def _strategy_means(pattern, tag_names):
        _, _, _, _, counts_per_trace = compute_trace_stats(pattern, tag_names)
        return {tag: (float(np.mean(counts_per_trace[tag])) if counts_per_trace[tag] else 0.0)
                for tag in tag_names}
    strategy_means = {}
    for model_name, r_stem, c_stem in MODELS:
        pretty_model = model_name.replace('\\', '')
        for regime, stem in [("cot", c_stem), ("reasoning", r_stem)]:
            for dataset, folder, tag_names in [
                ("Eedi", "eedi_data", EEDI_TAG_NAMES),
                ("SciQ", "sciq_data", SCIQ_TAG_NAMES),
            ]:
                pattern = f"{folder}/joint_results/annotated/{stem}_*_annot_parsed.csv"
                strategy_means[(pretty_model, regime, dataset)] = _strategy_means(pattern, tag_names)

models   = sorted({k[0] for k in strategy_means})
datasets = ["Eedi", "SciQ"]
strategies = [tag for tag, _ in ROW_ORDER]

# ratios[model][strategy] = [ratio_Eedi, ratio_SciQ]
ratios = {m: {} for m in models}
for m in models:
    for s in strategies:
        per_ds = []
        for d in datasets:
            r_mean = strategy_means[(m, "reasoning", d)].get(s, 0.0)
            c_mean = strategy_means[(m, "cot", d)].get(s, 0.0)
            if c_mean <= EPS and r_mean <= EPS:
                per_ds.append(float("nan"))  # 0/0 -> undefined
            else:
                per_ds.append(r_mean / max(c_mean, EPS))
        ratios[m][s] = per_ds


def _fmt(x):
    return "nan" if (isinstance(x, float) and np.isnan(x)) else f"{x:.2f}"


print("Per-strategy reasoning/CoT ratios (avg across datasets), per model")
print("Welch's t-test compares the dataset-level ratios (n=2 per model)")
print("-" * 78)
print(f"{'Strategy':<24} " + "  ".join(f"{m:>18}" for m in models) +
      f"  {'|DS-GLM|':>9}  {'p (Welch)':>10}")
print("-" * 78)

for s in strategies:
    cells = []
    means_per_model = {}
    for m in models:
        arr = np.array(ratios[m][s], dtype=float)
        finite = arr[np.isfinite(arr)]
        mean = float(np.mean(finite)) if finite.size else float("nan")
        means_per_model[m] = mean
        if finite.size > 1:
            ci_half = (float(np.std(finite, ddof=1)) *
                       _sps.t.ppf(0.975, finite.size - 1) / np.sqrt(finite.size))
        else:
            ci_half = 0.0
        cells.append(f"{_fmt(mean):>7} +/- {ci_half:>5.2f}  (n={finite.size})")

    if len(models) == 2:
        a = np.array(ratios[models[0]][s], dtype=float)
        b = np.array(ratios[models[1]][s], dtype=float)
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if a.size >= 2 and b.size >= 2 and (np.std(a) + np.std(b)) > 0:
            t_res = _sps.ttest_ind(a, b, equal_var=False)
            pval = float(t_res.pvalue)
        else:
            pval = float("nan")
        diff = abs(means_per_model[models[0]] - means_per_model[models[1]])
        diff_str = f"{diff:>9.2f}"
        pval_str = ("nan" if np.isnan(pval) else f"{pval:.3f}").rjust(10)
    else:
        diff_str, pval_str = "", ""

    print(f"{STRATEGY_DISPLAY.get(s, s):<24} " + "  ".join(c.rjust(18) for c in cells) +
          f"  {diff_str}  {pval_str}")

print()
print("Note: with only 2 datasets per model, Welch's t-test has very little power;")
print("treat p-values as descriptive. Look at the |DS-GLM| column for magnitude.")


In [ ]:
# Pairwise row-wise top-3 dominant-agreement (>=15% outgoing mass) across all
# (model, regime, dataset) settings, plus aggregates that group pairs by which
# dimension(s) differ between the two compared settings.

from itertools import combinations

K = 3
THRESH = 0.15

# Build the transition Counters once per (model, regime, dataset).
# Tag-name set depends on dataset.
SETTING_TO_PATTERN = {}
SETTING_TO_TAGS = {}
for model_name, r_stem, c_stem in MODELS:
    pretty_model = model_name.replace('\\', '')
    for regime, stem in [("cot", c_stem), ("reasoning", r_stem)]:
        for dataset, folder, tag_names in [
            ("Eedi", "eedi_data", EEDI_TAG_NAMES),
            ("SciQ", "sciq_data", SCIQ_TAG_NAMES),
        ]:
            key = (pretty_model, regime, dataset)
            SETTING_TO_PATTERN[key] = f"{folder}/joint_results/annotated/{stem}_*_annot_parsed.csv"
            SETTING_TO_TAGS[key] = tag_names

transitions_by_setting = {
    key: get_row_wise_transitions(pat, SETTING_TO_TAGS[key])
    for key, pat in SETTING_TO_PATTERN.items()
}

settings = list(transitions_by_setting.keys())


def _agreement(key_a, key_b):
    # Intersection of tag-name sets across the two datasets.
    common_tags = set(SETTING_TO_TAGS[key_a]) & set(SETTING_TO_TAGS[key_b])
    return top_k_rowwise_dominant_agreement(
        transitions_by_setting[key_a],
        transitions_by_setting[key_b],
        k=K, threshold=THRESH,
    )


# ---- Full pairwise table ----
pairs = list(combinations(settings, 2))
pair_scores = [(a, b, _agreement(a, b)) for a, b in pairs]


def _label(key):
    m, r, d = key
    return f"{m}/{r}/{d}"


print(f"Pairwise row-wise top-{K} dominant agreement (>= {THRESH} outgoing mass)")
print("-" * 78)
print(f"{'Setting A':<30} {'Setting B':<30}  Agreement")
print("-" * 78)
for a, b, score in pair_scores:
    print(f"{_label(a):<30} {_label(b):<30}  {score:.3f}")


# ---- Aggregations by which dimensions vary across the pair ----
def _varies(a, b):
    return (a[0] != b[0], a[1] != b[1], a[2] != b[2])  # (model, regime, dataset)


GROUP_LABELS = {
    (True,  False, False): "model differs only",
    (False, True,  False): "regime differs only",
    (False, False, True):  "dataset differs only",
    (True,  True,  False): "model + regime differ",
    (True,  False, True):  "model + dataset differ",
    (False, True,  True):  "regime + dataset differ",
    (True,  True,  True):  "all three differ",
}

print()
print("-" * 78)
print("Aggregated agreement by which dimension(s) differ across the pair")
print("(mean +/- 95% CI across pairs in the group)")
print("-" * 78)
print(f"{'Group':<30} {'n':>3}  mean +/- 95% CI")
print("-" * 78)


def _summary(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size > 1:
        mean, ci = get_mean_and_ci(arr)
    elif arr.size == 1:
        mean, ci = float(arr[0]), 0.0
    else:
        mean, ci = float('nan'), 0.0
    return arr.size, mean, ci


for variation, label in GROUP_LABELS.items():
    scores = [s for a, b, s in pair_scores if _varies(a, b) == variation]
    n, mean, ci = _summary(scores)
    print(f"{label:<30} {n:>3}  {mean:.3f} +/- {ci:.3f}")


# ---- Slice aggregates that hold one factor constant ----
print()
print("-" * 78)
print("Aggregated agreement holding one dimension constant, varying the others")
print("(mean +/- 95% CI across all pairs sharing the constant)")
print("-" * 78)

for dim, name in [(0, "model"), (1, "regime"), (2, "dataset")]:
    values_by_level = {}
    for a, b, s in pair_scores:
        if a[dim] == b[dim]:
            values_by_level.setdefault(a[dim], []).append(s)
    for level, scores in values_by_level.items():
        n, mean, ci = _summary(scores)
        print(f"{(name + '=' + level):<30} {n:>3}  {mean:.3f} +/- {ci:.3f}")


# ---- "Same regime" / "same dataset" / "same model" cross-only (compares the
#       counterpart in the OTHER value of every non-fixed dim — classic
#       cross-model agreement etc.) ----
print()
print("-" * 78)
print("Classic cross-X agreement (varies only X, all other dims held equal)")
print("-" * 78)
for variation, label in [
    ((True,  False, False), "cross-model (regime,dataset fixed)"),
    ((False, True,  False), "cross-regime (model,dataset fixed)"),
    ((False, False, True),  "cross-dataset (model,regime fixed)"),
]:
    scores = [s for a, b, s in pair_scores if _varies(a, b) == variation]
    n, mean, ci = _summary(scores)
    print(f"{label:<40} {n:>3}  {mean:.3f} +/- {ci:.3f}")


# ---- Overall ----
n, mean, ci = _summary([s for _, _, s in pair_scores])
print()
print("-" * 78)
print(f"{'Overall (all pairs)':<30} {n:>3}  {mean:.3f} +/- {ci:.3f}")


Next cell -> Table 32 (Appendix D.1, "Coverage of Strategies").

In [ ]:
# Same table layout, but reporting the percentage of traces containing at least
# one occurrence of each strategy.

def _presence_for(pattern, tag_names):
    _, _, _, presence_pct, _ = compute_trace_stats(pattern, tag_names)
    return {tag: float(presence_pct.get(tag, 0.0)) for tag in tag_names}


presence = {}
for model_name, r_stem, c_stem in MODELS:
    presence[model_name] = {
        ("cot", "Eedi"):       _presence_for(f"eedi_data/joint_results/annotated/{c_stem}_*_annot_parsed.csv", EEDI_TAG_NAMES),
        ("cot", "SciQ"):       _presence_for(f"sciq_data/joint_results/annotated/{c_stem}_*_annot_parsed.csv", SCIQ_TAG_NAMES),
        ("reasoning", "Eedi"): _presence_for(f"eedi_data/joint_results/annotated/{r_stem}_*_annot_parsed.csv", EEDI_TAG_NAMES),
        ("reasoning", "SciQ"): _presence_for(f"sciq_data/joint_results/annotated/{r_stem}_*_annot_parsed.csv", SCIQ_TAG_NAMES),
    }


def _split1(x):
    a, b = f"{x:.1f}".split(".")
    return a, b


def _fmt_pct(p):
    i, d = _split1(p)
    return f"{i} & {d}"


header_pct = r"""\begin{table*}[!htbp]
\centering
\small
\begin{tabular}{ll|
                r@{.}l r@{.}l
                r@{.}l r@{.}l}
\hline
& &
\multicolumn{4}{c}{\textbf{CoT}}
& \multicolumn{4}{c}{\textbf{Reasoning}} \\
\cmidrule(lr){3-6}
\cmidrule(lr){7-10}

\textbf{Model} & \textbf{Strategy} &
\multicolumn{2}{c}{\textbf{Eedi}}
& \multicolumn{2}{c}{\textbf{SciQ}}
& \multicolumn{2}{c}{\textbf{Eedi}}
& \multicolumn{2}{c}{\textbf{SciQ}} \\
\hline
"""

footer_pct = r"""\hline
\end{tabular}
\caption{Percentage of traces containing at least one occurrence of each taxonomy strategy (\cref{tab:taxonomy-definition}) for each model on both datasets.}
\label{tab:label_presence_comparison}
\end{table*}"""

body_blocks_pct = []
for model_name, _, _ in MODELS:
    rows = []
    for tag, display in ROW_ORDER:
        cells = []
        for trace_type in ("cot", "reasoning"):
            for dataset in ("Eedi", "SciQ"):
                cells.append(_fmt_pct(presence[model_name][(trace_type, dataset)][tag]))
        rows.append(f"& {display}\n& " + "\n& ".join(cells) + r" \\")
    block = f"\\multirow{{10}}{{*}}{{\\textbf{{{model_name}}}}}\n\n" + "\n\n".join(rows)
    body_blocks_pct.append(block)

body_pct = "\n\n\\hline\n\n".join(body_blocks_pct)

print(header_pct + body_pct + "\n\n" + footer_pct)

In [ ]:
# Overall average trace length in the reasoning setting (across both datasets and both models).
reasoning_patterns = [
    ("eedi_data/joint_results/annotated/deepseek-naive-deepseek-reasoner_*_annot_parsed.csv", EEDI_TAG_NAMES),
    ("eedi_data/joint_results/annotated/openrouter-naive-z-ai_glm-4.7-reasoner_*_annot_parsed.csv", EEDI_TAG_NAMES),
    ("sciq_data/joint_results/annotated/deepseek-naive-deepseek-reasoner_*_annot_parsed.csv", SCIQ_TAG_NAMES),
    ("sciq_data/joint_results/annotated/openrouter-naive-z-ai_glm-4.7-reasoner_*_annot_parsed.csv", SCIQ_TAG_NAMES),
]

total_traces = 0
total_chars = 0
for pattern, tag_names in reasoning_patterns:
    n, avg_len, _, _, _ = compute_trace_stats(pattern, tag_names)
    total_traces += n
    total_chars += n * avg_len

overall_avg = total_chars / total_traces if total_traces > 0 else 0
print(f"Overall average reasoning trace length: {overall_avg:.1f} chars (n={total_traces})")